# soft_prompt_ii_train_p1247 — variant (ii), conditional run at P = 1,247
Trains the soft prompt for variant (ii): P learnable rows appended to the MLLM's output, i.e. to the conditioning stream the diffusion transformer attends to. Everything else is frozen. OmniGen2 ships no soft-prompt recipe, so the loop is a patch onto `train.py`, applied with the same anchored `src.replace` pattern the LoRA notebooks use and validated by `soft_prompt_ii_probe_p668`.

Measured by the probe: 12.14 s/step at P=64 and 15.06 s/step at P=668, peak 27.2 GB, and a usable learning rate of 1e-3. Step cost at P=1,247 is NOT measured — the joint sequence the diffusion transformer attends over grows 11% to 19% depending on reference count (3,016-5,064 tokens at P=668 against 3,595-5,643 at P=1,247) and attention is quadratic in it, so re-run soft_prompt_ii_probe_p668 section 10 with P = 1247 before scheduling. Prompt drift was +3.6% at both 64 and 668, which supports 1e-3 as the candidate rate for P=1,247; the probe should confirm it alongside the cost — roughly 1,250x LoRA's 8e-7. The P=64 figure predates the loss-normalisation correction; at this variant's gradient magnitude (~6e-3) AdamW's approximate scale invariance is expected to make the difference negligible, and the P=668 drift reproduced to 0.02 pp across two runs. Both arms apply the same loss normalisation (`FIX_ACCUM_NORM`), so the ratio is a difference in learning rate rather than in gradient scale. Learning rates at or above 1e-2 blow the prompt norm out by 2.5x to 22x while the gradient collapses by one to two orders of magnitude.

3000 steps, checkpoints written every 500. Writing a checkpoint is free (the prompt is a few MB); scoring one is not, so only four are scored later, at 500 / 1000 / 2000 / 3000. Selection is argmax of validation Overall over the scored checkpoints. If step 3000 takes the argmax, schedule a fresh run from step 0 with a longer budget rather than resuming: the prompt sits outside the accelerate state, so `prompt-3000.pt` cannot continue the trajectory under the current implementation.

This notebook is the **P = 1,247** run: 1247 x 2048 = 2,553,856 trainable, 100.01% of LoRA rank 4, the nearest-integer parameter match. It is CONDITIONAL — run only if (ii) at P=668 lands below LoRA r4 @1000 on the 488-row test split with the 90% CI excluding -0.15, i.e. only if capacity is a live alternative explanation for the deficit. Validation loss is logged every 250 steps on the same 98-row slice, with the same repeat count and base seed as the LoRA monitor. The draws themselves are not matched: the evaluator seeds on the batch start index, and this run uses validation batch 1 to keep the injection unpadded while the executed arms used batch 2. Not as an early-stopping signal — the LoRA arm established it runs opposite to the task metric — but as an internal within-run monitor, where the same qualitative inversion would be corroborating evidence rather than a controlled cross-arm replication.

## 1. Repo, deps

In [ ]:
import os, subprocess, sys
REPO = "/content/OmniGen2"
# project revision repeatedly recorded across the workflow by inspect_official_training,
# inspect_soft_prompt_surface, inspect_quantization_surface and both qlora_quant_probes;
# unpinned historical clones prevent proving it for every completed run.
COMMIT = "18e6f9d5271b517fcb32e999f10df943ae9b8f20"
if not os.path.isdir(REPO):
    subprocess.run(["git","clone","https://github.com/VectorSpaceLab/OmniGen2.git",REPO], check=True)
else:
    print("OmniGen2 already present.")

# establish the revision first, and restore every tracked file to it. HEAD equality alone would
# accept a working tree still carrying another variant's train.py patch: four of the five patch
# guards below use variant-agnostic sentinels, so a stale patch would be silently kept.
subprocess.run(["git","-C",REPO,"fetch","-q","--depth","50","origin",COMMIT], check=False)
subprocess.run(["git","-C",REPO,"reset","-q","--hard",COMMIT], check=True)
_head = subprocess.run(["git","-C",REPO,"rev-parse","HEAD"], capture_output=True, text=True).stdout.strip()
assert _head == COMMIT, f"repo is at {_head}, expected {COMMIT}"
_dirty = subprocess.run(["git","-C",REPO,"status","--porcelain","--untracked-files=no"],
                        capture_output=True, text=True).stdout.strip()
assert not _dirty, f"tracked files still modified after reset:\n{_dirty}"

# install against the pinned revision, on both paths
subprocess.run([sys.executable,"-m","pip","install","-q","-r","requirements.txt"], cwd=REPO, check=True)
subprocess.run([sys.executable,"-m","pip","install","-q","-U","huggingface_hub[hf_xet]>=0.30.0,<1.0"], check=True)
# requirements.txt omits torchdiffeq (omnigen2/transport/integrators.py) and peft (train.py)
subprocess.run([sys.executable,"-m","pip","install","-q","torchdiffeq","peft"], check=True)

# versions are logged, not pinned: the completed runs did not record theirs, so a pin would guess
_vers = subprocess.run([sys.executable,"-c",
                        "import peft, torchdiffeq, torch; "
                        "print(f'peft {peft.__version__} torchdiffeq {torchdiffeq.__version__} torch {torch.__version__}')"],
                       capture_output=True, text=True)
if _vers.returncode != 0:
    raise RuntimeError(f"a required package is not importable; train.py needs peft and torchdiffeq.\n{_vers.stderr.strip()[-400:]}")
print(f"repo at {_head[:7]}, clean tree | {_vers.stdout.strip()}")

# peft's is_torchao_available() raises rather than returning False when torchao predates 0.16;
# Colab ships 0.10. torchao is unused by this bf16 path. Guarded, so a no-op once gone.
_ao = subprocess.run([sys.executable,"-c",
                      "from peft.import_utils import is_torchao_available; is_torchao_available()"],
                     capture_output=True, text=True)
if _ao.returncode != 0:
    subprocess.run([sys.executable,"-m","pip","uninstall","-y","-q","torchao"], check=True)
    print("torchao removed")
sys.path.insert(0, REPO); os.chdir(REPO)
print(subprocess.run(["nvidia-smi","--query-gpu=name,memory.total","--format=csv,noheader"],
                     capture_output=True, text=True).stdout.strip())

## 2. Drive, images, splits

In [ ]:
from google.colab import drive, userdata
import json
drive.mount("/content/drive")
os.environ["HF_TOKEN"] = userdata.get("HF_TOKEN")

DRIVE = "/content/drive/MyDrive/Colab Notebooks/My Master's Thesis/Data"
IMGS, EXPECTED = "/content/imgs", 37063
TAR = f"{DRIVE}/imgs_subset.tar"

def n_files(d):
    return sum(len(f) for _, _, f in os.walk(d)) if os.path.isdir(d) else 0

if n_files(IMGS) < EXPECTED:
    os.makedirs(IMGS, exist_ok=True)
    subprocess.run(f'cp "{TAR}" /content/imgs_subset.tar && tar -xf /content/imgs_subset.tar -C {IMGS}',
                   shell=True, check=True)
    print("restored", n_files(IMGS), "/", EXPECTED)
    assert n_files(IMGS) >= EXPECTED, "restore incomplete"
else:
    print("images complete:", n_files(IMGS))

for s in ["train", "val"]:
    src = f"{DRIVE}/{s}.jsonl"
    if not os.path.exists(f"/content/{s}.jsonl"):
        subprocess.run(["cp", src, f"/content/{s}.jsonl"], check=True)
subprocess.run(["cp", f"{DRIVE}/screen_exclude_from_training.txt", "/content/"], check=True)
print("splits restored")

## 3. Prompt length, initialisation, run identity
`P` is the number of learnable rows; each is 2048 wide, the MLLM's hidden size, so trainable parameters are `P x 2048`.

`P = 668` is the primary size for this variant: it keeps instruction plus prompt inside the 888-token length the decoder was trained on for 96.23% of training rows, reaching 956 rows at the longest, both measured on the chat-templated string at the constant 25-token overhead, and matched to variant (i) so that the comparison was intended to isolate injection site from capacity, which the known variant-(ii) padding defect compromises in the executed artifact. `P = 1,247` is this notebook, the nearest-integer parameter match against LoRA rank 4 at 100.01%. No fundamental architectural cap prevents it once the diffusion RoPE table is extended — variant (ii) appends after the MLLM, so neither the 888-token collator limit nor `max_sequence_length` applies, though the released 1024-entry axis-0 table would overflow on every row and is extended to 4096 in section 7 — but it puts the conditioning stream at 1,535 rows against the 888 seen in training, from the 288-token templated maximum of the 263-token longest training instruction section 4 measures, so it trades a capacity confound for an out-of-distribution one and is run only as a capacity control.

Rows are initialised from the embeddings of real domain words rather than noise. Lester et al. (2021) evaluate sampled-vocabulary initialisation alongside random-uniform and class-label initialisation; no alternative initialisation is run here, so this project makes no claim about its effect.

In [ ]:
import torch, re, glob
from huggingface_hub import hf_hub_download
from safetensors.torch import load_file
from transformers import AutoTokenizer

P    = 1247       # 1247 x 2048 = 2,553,856 = 100.01% of LoRA r4. Conditional run: only if
                  # (ii) P=668 lands below LoRA r4 @1000 with the 90% CI excluding -0.15
LR   = 1e-3       # from soft_prompt_ii_probe_p668, chosen on prompt drift; the gradient norm is the
                  # supporting check — it peaks here and collapses one to two orders above
STEPS = 3000
CKPT_EVERY = 500
SEED = 2233
MODEL = "OmniGen2/OmniGen2"
# the completed runs did not record a base-model revision and it cannot be recovered, so this
# resolves the current one, uses it for every load below, and records it in the manifest.
# This pins the notebook-side tokenizer and embedding reads only. train.py resolves the base
# model through its own config, so the weights the trainer loads are not yet tied to this
# revision; that has to be closed before the run if snapshot reproducibility is wanted.
# It makes no claim about the historical snapshot either way.
from huggingface_hub import HfApi
MODEL_REV = HfApi().model_info(MODEL).sha
print(f"base model {MODEL} @ {MODEL_REV}")
RUN = f"sp_ii_p{P}"

torch.manual_seed(SEED)
tok = AutoTokenizer.from_pretrained(MODEL, subfolder="processor", trust_remote_code=True, revision=MODEL_REV)

# read only the embedding matrix out of the mllm shards, not the whole 3B model
idx = json.load(open(hf_hub_download(MODEL, "mllm/model.safetensors.index.json", revision=MODEL_REV)))
key = next(k for k in idx["weight_map"] if k.endswith("embed_tokens.weight"))
emb = load_file(hf_hub_download(MODEL, f"mllm/{idx['weight_map'][key]}", revision=MODEL_REV))[key]

SEED_WORDS = ("a photo of a laptop tablet smartphone headphones earbuds keyboard computer mouse "
              "game controller gaming console desktop computer setup vr headset on a wooden desk "
              "in an office studio lighting product shot detailed realistic")
ids  = tok(SEED_WORDS, add_special_tokens=False).input_ids
init = emb[torch.tensor((ids * ((P // len(ids)) + 1))[:P])].clone().float()
assert init.shape == (P, emb.shape[1]), init.shape

EXP = f"{DRIVE}/soft_prompt/experiments/{RUN}"
os.makedirs(EXP, exist_ok=True)
_stale = sorted(glob.glob(f"{EXP}/prompt-*.pt")) + [
    x for x in (f"{EXP}/val_loss.jsonl", f"{EXP}/manifest.json", f"{EXP}/console.log")
    if os.path.exists(x)]
if _stale:
    raise RuntimeError(
        f"run directory already holds artifacts: {[os.path.basename(x) for x in _stale]}. "
        f"val_loss.jsonl appends and prompt-*.pt overwrite per step, so a restart here can mix "
        f"launches. Use a fresh RUN, or archive {EXP} deliberately before rerunning.")
os.makedirs("/content/sp", exist_ok=True)
torch.save(init, "/content/sp/init.pt")
json.dump({"P": P, "dim": int(emb.shape[1]), "init_path": "/content/sp/init.pt",
           "seed": SEED, "save_dir": EXP, "every": CKPT_EVERY},
          open("/content/soft_prompt.json", "w"))

LORA_R4 = 38 * 16800 * 4
print(f"run {RUN}  P={P}  dim={emb.shape[1]}  trainable={init.numel():,} "
      f"= {100*init.numel()/LORA_R4:.0f}% of LoRA r4")
print(f"lr {LR}  steps {STEPS}  checkpoints every {CKPT_EVERY} -> {EXP}")
print(f"init norm {init.norm():.3f}  (probe start values: 6.380 at P=64, 20.610 at P=668)")

## 4. Training data
The full domain training split with the 62 screen targets excluded, in OmniGen2's format. Identical row set to the LoRA arm.

In [ ]:
rows = [json.loads(l) for l in open("/content/train.jsonl")]
excl = {l.strip() for l in open("/content/screen_exclude_from_training.txt") if l.strip()}
rows = [r for r in rows if r["output_image"] not in excl]
assert len(rows) == 39591, f"expected 39591 training rows, got {len(rows)}"
from transformers import AutoTokenizer
_tok = AutoTokenizer.from_pretrained(MODEL, subfolder="processor", trust_remote_code=True, revision=MODEL_REV)
# the collator truncates the chat-templated string, so the budget is measured on that.
# SYS is OmniGen2TrainDataset.SYSTEM_PROMPT; the overhead is a constant 25 tokens.
SYS = "You are a helpful assistant that generates high-quality images based on user instructions."
_tpl = lambda s: _tok.apply_chat_template(
    [{"role": "system", "content": SYS}, {"role": "user", "content": s}],
    tokenize=False, add_generation_prompt=False)
_raw = [len(x) for x in _tok([r["instruction"] for r in rows], add_special_tokens=False).input_ids]
_len = [len(x) for x in _tok([_tpl(r["instruction"]) for r in rows], add_special_tokens=False).input_ids]
MAX_INSTR = max(_len)
print(f"longest training instruction {MAX_INSTR} templated / {max(_raw)} raw tokens; "
      f"prompt {P} + {MAX_INSTR} = {P + MAX_INSTR} conditioning rows")
print(f"{100*sum(1 for n in _len if P + n <= 888)/len(_len):.2f}% of training rows keep prompt+instruction inside the 888 seen in training")
# not a hard cap: the prompt is injected after the collator truncates the instruction, so this reports distribution, not a limit
print(f"train rows {len(rows)}  (39,893 minus the 302 screen rows)")

os.makedirs("/content/data", exist_ok=True)
with open("/content/data/train_sp.jsonl", "w") as f:
    for r in rows:
        f.write(json.dumps({
            "task_type": "ic",
            "instruction": r["instruction"],
            "input_images": [os.path.join(IMGS, p) for p in r["input_images"]],
            "output_image": os.path.join(IMGS, r["output_image"]),
        }) + "\n")

os.makedirs(f"{REPO}/data_configs/train/sp", exist_ok=True)
with open(f"{REPO}/data_configs/train/sp/sp.yml", "w") as f:
    f.write("data:\n  -\n    path: /content/data/train_sp.jsonl\n    type: 'ic'\n    ratio: !!float 1\n")
print("data config written")

## 5. Patch train.py — freeze everything, train the prompt only
The two edits validated by the probe, each asserted to match exactly once.

**A** replaces the optimizer's parameter list: the transformer, VAE and text encoder are frozen and the optimizer is given the prompt alone. `accelerator.clip_grad_norm_` at line 592 already takes `trainable_params`, so clipping follows.

**B** appends the prompt to `text_feats` immediately after the `torch.no_grad()` block that produces it, and extends `text_attention_mask` to match. Placement after the block is what makes the prompt trainable: the encoder output is detached, the prompt is not. The probe confirmed a non-zero gradient reaches it.

In [ ]:
p = f"{REPO}/train.py"
src = open(p, encoding="utf-8").read()

A = "    trainable_params = list(filter(lambda p: p.requires_grad, model.parameters()))\n"
A_NEW = "\n".join([
    "    # soft prompting variant (ii): the only trainable tensor is a P x hidden prompt appended to",
    "    # the MLLM output. Transformer, VAE and text encoder are frozen. OmniGen2 ships no soft-prompt",
    "    # path, so this replaces the LoRA/full-FT parameter selection rather than extending it.",
    "    import json as _json",
    "    _spc = _json.load(open('/content/soft_prompt.json'))",
    "    model.requires_grad_(False)",
    "    SOFT_PROMPT = torch.nn.Parameter(",
    "        torch.load(_spc['init_path'], map_location='cpu').to(accelerator.device, dtype=torch.float32))",
    "    SOFT_PROMPT.requires_grad_(True)",
    "    globals()['SOFT_PROMPT'] = SOFT_PROMPT",
    "    globals()['SP_CFG'] = _spc",
    "    trainable_params = [SOFT_PROMPT]",
    "    logger.info('[soft_prompt] variant(ii) P=%d dim=%d trainable=%d' % (",
    "        SOFT_PROMPT.shape[0], SOFT_PROMPT.shape[1], SOFT_PROMPT.numel()))",
]) + "\n"

B = "                    ).last_hidden_state\n"
B_NEW = "\n".join([
    "                # appended OUTSIDE the no_grad block above: text_feats is detached, SOFT_PROMPT is",
    "                # not, so the loss reaches the prompt through the diffusion transformer",
    "                if int(text_mask.sum()) != text_mask.numel():",
    "                    raise RuntimeError('padded batch: appending after padding breaks the mask prefix - run batch_size 1')",
    "                _sp = SOFT_PROMPT.to(text_feats.dtype).unsqueeze(0).expand(text_feats.shape[0], -1, -1)",
    "                text_feats = torch.cat([text_feats, _sp], dim=1)",
    "                text_mask = torch.cat([text_mask,",
    "                    torch.ones(text_mask.shape[0], _sp.shape[1],",
    "                               dtype=text_mask.dtype, device=text_mask.device)], dim=1)",
    "                if not text_feats.requires_grad:",
    "                    raise RuntimeError('soft prompt is not in the graph - check patch B placement')",
]) + "\n"

# sentinel must be variant-specific: '[soft_prompt]' appears in variants (i) and (iii) too, so a
# train.py already patched for another variant in this runtime would skip this patch silently
if "variant(ii)" in src:
    print("already patched")
else:
    assert src.count(A) == 1, f"anchor A found {src.count(A)} times"
    assert src.count(B) == 1, f"anchor B found {src.count(B)} times"
    open(p, "w", encoding="utf-8").write(src.replace(A, A_NEW).replace(B, B + B_NEW))
    import ast; ast.parse(open(p, encoding="utf-8").read())
    print("patched and parses")

# repo.py writes text positions into all three rope axes, and the tables are sized from the
# released config. With P appended to the conditioning stream, cap_seq_len is real_text_len + P,
# which indexes past the table on the longest rows. get_1d_rotary_pos_embed is outer(arange(e),
# freqs), so row k is identical for any e > k: enlarging the table is a pure extension.
# The released config bounds axis 0 at 1024: P=668 exceeds it on the longest rows, and the
# parameter-matched P=1247 would exceed it on every row. Extending the table lifts that bound;
# the remaining concern at P=1247 is conditioning length against the 888 rows seen in training.
ROPE_LEN = 4096   # covers the 888 collator cap plus the largest P under test

R = """    freqs_cis = OmniGen2RotaryPosEmbed.get_freqs_cis(
        model.config.axes_dim_rope,
        model.config.axes_lens,
        theta=10000,
    )
"""
R_NEW = f"""    _axes_lens = tuple(max(int(l), {ROPE_LEN}) for l in model.config.axes_lens)
    logger.info("[soft_prompt] rope axes_lens %s -> %s" % (tuple(model.config.axes_lens), _axes_lens))
    freqs_cis = OmniGen2RotaryPosEmbed.get_freqs_cis(
        model.config.axes_dim_rope,
        _axes_lens,
        theta=10000,
    )
"""

src = open(p, encoding="utf-8").read()
if "_axes_lens" in src:
    print("rope tables already extended")
else:
    assert src.count(R) == 1, "rope anchor R not found exactly once"
    src = src.replace(R, R_NEW)
    ast.parse(src)
    open(p, "w", encoding="utf-8").write(src)
    print(f"rope tables extended to {ROPE_LEN}")

## 5a. Loss normalisation
gradient_state.num_steps is 1 on the reference 8-GPU config and equal to the accumulation count on
one GPU — 16 here — so the shipped normalisation runs at that many times the reference gradient
scale. The replacement drops the term entirely, so the correction does not depend on the
accumulation count. Corrected to match the
LoRA arm, which applies the same replacement. Derivation in lora_train_r{4,8,16} section 5.

In [ ]:
FIX_ACCUM_NORM = True

p = f"{REPO}/train.py"
A4     = ("                loss = (loss * accelerator.gradient_state.num_steps"
          " * accelerator.num_processes) / num_tokens_in_batch\n")
A4_NEW = "                loss = (loss * accelerator.num_processes) / num_tokens_in_batch\n"

src = open(p, encoding="utf-8").read()
if A4_NEW in src:
    print("accum-norm already corrected")
elif not FIX_ACCUM_NORM:
    print("FIX_ACCUM_NORM=False - shipped normalisation retained "
          "(scales with gradient_accumulation_steps)")
else:
    assert src.count(A4) == 1, f"anchor A4 found {src.count(A4)} times"
    open(p, "w", encoding="utf-8").write(src.replace(A4, A4_NEW))
    import ast; ast.parse(open(p, encoding="utf-8").read())
    print("accum-norm corrected and train.py parses")

now = open(p, encoding="utf-8").read()
print(f"  accum-norm fix : {A4_NEW in now}  (FIX_ACCUM_NORM={FIX_ACCUM_NORM})")
print(f"  shipped line   : {A4 in now}")

## 6. Checkpointing and monitor
`accelerator.save_state` captures the model, and the prompt is deliberately not part of it, so **S** writes the prompt itself at every checkpoint interval. Each file is a few MB.

**M** prints the gradient norm and the prompt norm every 50 steps. Over a long run this is the live answer to whether the arm is learning: a gradient that dies or a prompt norm that stops moving flags optimisation saturation or a stall, to be read against the loss curve rather than taken as an expressiveness limit. The probe measured gradient 1.4e-1 with prompt norm rising 6.380 -> 6.609 at P=64, and 6.184e-03 with 20.610 -> 21.346 at P=668, both over 15 steps at lr 1e-3; expect the same shape at this P with a proportionally larger starting norm.

In [ ]:
src = open(p, encoding="utf-8").read()

S = "                if global_step % args.logger.checkpointing_steps == 0:\n"
S_NEW = "\n".join([
    "                if global_step % SP_CFG['every'] == 0 and accelerator.is_main_process:",
    "                    import os as _os",
    "                    _d = SP_CFG['save_dir']",
    "                    _os.makedirs(_d, exist_ok=True)",
    "                    torch.save({'soft_prompt': SOFT_PROMPT.detach().cpu(),",
    "                                'step': global_step, 'P': SOFT_PROMPT.shape[0],",
    "                                'dim': SOFT_PROMPT.shape[1]},",
    "                               _os.path.join(_d, 'prompt-%d.pt' % global_step))",
    "                    logger.info('[sp] saved prompt-%d.pt' % global_step)",
    "                # S replaces the condition line only, so the shipped accelerator.save_state body",
    "                # below would otherwise run at every SP_CFG['every']. It cannot restore this arm:",
    "                # SOFT_PROMPT is a standalone Parameter, not part of the model, so a resume would",
    "                # restore weights, optimizer and step counter while re-initialising the prompt from",
    "                # init.pt. Dead-branched: the prompt tensor above is the checkpoint.",
        "                if False and global_step % args.logger.checkpointing_steps == 0:",
]) + "\n"

M = "                    accelerator.clip_grad_norm_(trainable_params, args.train.max_grad_norm)\n"
M_NEW = "\n".join([
    # clip_grad_norm_ returns the PRE-clip total norm; reading .grad after the call
    # returns the clipped value, which saturates at max_grad_norm and is not a measurement
    "                    _gnorm = accelerator.clip_grad_norm_(trainable_params, args.train.max_grad_norm)",
    "                    if accelerator.is_main_process and global_step % 50 == 0:",
    "                        print('[sp] step %d grad_norm %.3e prompt_norm %.4f' % (",
    "                            global_step, float(_gnorm),",
    "                            SOFT_PROMPT.detach().float().norm().item()), flush=True)",
]) + "\n"

if "[sp] saved" in src:
    print("already patched")
else:
    assert src.count(S) == 1, f"anchor S found {src.count(S)} times"
    assert src.count(M) == 1, f"anchor M found {src.count(M)} times"
    open(p, "w", encoding="utf-8").write(src.replace(S, S_NEW).replace(M, M_NEW))
    import ast; ast.parse(open(p, encoding="utf-8").read())
    print("checkpoint and monitor patched")

## 7. Validation loss module
The evaluator the LoRA arm used, plus an `encode_text` hook. Without it the val path would call the frozen text encoder directly and measure the model with no prompt attached — a flat curve that looks like a result.

In [ ]:
VAL_MODULE = '"""Held-out validation loss for OmniGen2 training.\n\nEvaluates a deterministic held-out proxy for the training objective, using the VAE posterior mode rather than a posterior sample so successive evaluations isolate weight changes, on a held-out split with gradients disabled.\nTimesteps and noise are drawn through the repository\'s own transport.sample under\na fixed seed, so the timestep distribution and resolution-dependent shift match\ntraining exactly while successive evaluations differ only by the model weights.\n\nGlobal RNG state is saved and restored around every evaluation so that the\ntraining stream is unaffected.\n\nNot part of the upstream OmniGen2 repository.\n"""\n\nimport json\nimport os\nimport time\n\nimport torch\n\n\nclass ValLossEvaluator:\n    """Per-token velocity MSE on a held-out split at fixed timesteps and noise."""\n\n    def __init__(self, dataset, collator, transport, out_path, batch_size=2,\n                 every=250, n_repeats=4, seed=1234, encode_text=None):\n        self.dataset = dataset\n        self.collator = collator\n        self.transport = transport\n        self.out_path = out_path\n        self.batch_size = batch_size\n        self.every = every\n        self.n_repeats = n_repeats\n        self.seed = seed\n        self.encode_text = encode_text\n        d = os.path.dirname(out_path)\n        if d:\n            os.makedirs(d, exist_ok=True)\n\n    @torch.no_grad()\n    def run(self, model, vae, text_encoder, freqs_cis, weight_dtype, global_step, device):\n        t_start = time.time()\n        was_training = model.training\n        model.eval()\n\n        cpu_rng = torch.get_rng_state()\n        cuda_rng = torch.cuda.get_rng_state_all() if torch.cuda.is_available() else None\n\n        sse = 0.0\n        tokens = 0.0\n        by_t = []\n\n        try:\n            for start in range(0, len(self.dataset), self.batch_size):\n                idx = list(range(start, min(start + self.batch_size, len(self.dataset))))\n                batch = self.collator([self.dataset[i] for i in idx])\n\n                text_ids = batch["text_ids"].to(device)\n                text_mask = batch["text_mask"].to(device)\n                if self.encode_text is None:\n                    text_feats = text_encoder(\n                        input_ids=text_ids,\n                        attention_mask=text_mask,\n                        output_hidden_states=False,\n                    ).last_hidden_state\n                else:\n                    # soft-prompt arms: mirror the training-loop injection so val loss is\n                    # measured on the same conditioning the optimiser is shaping\n                    text_feats, text_mask = self.encode_text(text_encoder, text_ids, text_mask)\n\n                def encode(img):\n                    z = vae.encode(img.to(device=device, dtype=vae.dtype)).latent_dist.mode()\n                    if vae.config.shift_factor is not None:\n                        z = z - vae.config.shift_factor\n                    if vae.config.scaling_factor is not None:\n                        z = z * vae.config.scaling_factor\n                    return z.to(dtype=weight_dtype)\n\n                ref_latents = []\n                for imgs in batch["input_images"]:\n                    if imgs is not None and len(imgs) > 0:\n                        ref_latents.append([encode(im).squeeze(0) for im in imgs])\n                    else:\n                        ref_latents.append(None)\n\n                x1 = [encode(im).squeeze(0) for im in batch["output_image"]]\n\n                model_kwargs = dict(\n                    text_hidden_states=text_feats,\n                    text_attention_mask=text_mask,\n                    ref_image_hidden_states=ref_latents,\n                    freqs_cis=freqs_cis,\n                )\n\n                for k in range(self.n_repeats):\n                    torch.manual_seed((self.seed * 100003 + start * 101 + k) % (2 ** 31 - 1))\n                    t, x0, _ = self.transport.sample(x1, 0, 1)\n                    t, xt, ut = self.transport.path_sampler.plan(t, x0, x1)\n                    pred = model(xt, t, **dict(model_kwargs))\n                    for j in range(len(x1)):\n                        d = (pred[j].float() - ut[j].float()) ** 2\n                        sse += float(d.sum().item())\n                        tokens += float(d.numel())\n                        by_t.append((float(t[j].item()), float(d.mean().item())))\n        finally:\n            torch.set_rng_state(cpu_rng)\n            if cuda_rng is not None:\n                torch.cuda.set_rng_state_all(cuda_rng)\n            if was_training:\n                model.train()\n\n        val_loss = sse / max(tokens, 1.0)\n\n        bins = [0.0] * 4\n        counts = [0] * 4\n        for t_val, mse in by_t:\n            b = min(int(t_val * 4), 3)\n            bins[b] += mse\n            counts[b] += 1\n        binned = {f"t_{i/4:.2f}_{(i+1)/4:.2f}": (bins[i] / counts[i] if counts[i] else None)\n                  for i in range(4)}\n\n        with open(self.out_path, "a", encoding="utf-8") as f:\n            f.write(json.dumps({\n                "step": int(global_step),\n                "val_loss": val_loss,\n                "n_rows": len(self.dataset),\n                "n_repeats": self.n_repeats,\n                "n_samples": len(by_t),\n                "seconds": round(time.time() - t_start, 1),\n                "by_timestep_quartile": binned,\n            }) + "\\n")\n\n        return val_loss\n'

path = f"{REPO}/omnigen2/val_loss.py"
open(path, "w", encoding="utf-8").write(VAL_MODULE)
import ast; ast.parse(open(path, encoding='utf-8').read())
print(f"wrote {path} ({len(VAL_MODULE)} chars) and it parses")

## 8. Validation loss patch
The 98-row val slice, written in the same form as the training pool, and the two anchors that install the evaluator. `_sp_encode` mirrors patch B exactly, so validation loss is measured on the conditioning the optimiser is actually shaping.

In [ ]:
# validation loss on the frozen 98-row val slice: same rows as the LoRA arm. The draws are
# not matched: the evaluator seeds on the batch start index and this run uses batch_size 1
# to keep the injection unpadded, while every executed arm used batch_size 2, so this curve
# is an internal monitor for this run and is not comparable across arms
val_rows = [json.loads(l) for l in open("/content/val.jsonl", encoding="utf-8")]
assert len(val_rows) == 98, f"expected 98 val rows, got {len(val_rows)}"
assert not ({r["output_image"] for r in rows} & {r["output_image"] for r in val_rows}), \
    "val targets leaked into training"

with open("/content/data/val_sp.jsonl", "w", encoding="utf-8") as f:
    for r in val_rows:
        f.write(json.dumps({
            "task_type": "ic",
            "instruction": r["instruction"],
            "input_images": [os.path.join(IMGS, p) for p in r["input_images"]],
            "output_image": os.path.join(IMGS, r["output_image"]),
        }) + "\n")
with open(f"{REPO}/data_configs/train/sp/val_sp.yml", "w") as f:
    f.write("data:\n  -\n    path: /content/data/val_sp.jsonl\n    type: 'ic'\n    ratio: !!float 1\n")
print(f"val config written: {len(val_rows)} rows")

src = open(p, encoding="utf-8").read()
A1 = '    logger.info(f"{args.train.batch_size=} {args.train.gradient_accumulation_steps=} {accelerator.num_processes=} {args.train.global_batch_size=}")\n'
A2 = "                global_step += 1\n"

SETUP = '''    # held-out validation loss (not part of upstream OmniGen2)
    _val_eval = None
    if args.get("val_loss", None) is not None and accelerator.is_main_process:
        from omnigen2.val_loss import ValLossEvaluator
        _val_dataset = OmniGen2TrainDataset(
            args.val_loss.data_path,
            tokenizer=text_tokenizer,
            use_chat_template=args.data.use_chat_template,
            prompt_dropout_prob=0.0,
            ref_img_dropout_prob=0.0,
            max_input_pixels=OmegaConf.to_object(args.data.get("max_input_pixels", 1024 * 1024)),
            max_output_pixels=args.data.get("max_output_pixels", 1024 * 1024),
            max_side_length=args.data.get("max_side_length", 2048),
        )
        # variant (ii): prompt appended to the MLLM output
        def _sp_encode(enc, ids, mask):
            if int(mask.sum()) != mask.numel():
                raise RuntimeError("padded batch: appending after padding breaks the mask prefix - run batch_size 1")
            _f = enc(input_ids=ids, attention_mask=mask,
                     output_hidden_states=False).last_hidden_state
            _sp = SOFT_PROMPT.to(_f.dtype).unsqueeze(0).expand(_f.shape[0], -1, -1)
            _mask = torch.cat([mask, torch.ones(mask.shape[0], _sp.shape[1],
                                                dtype=mask.dtype, device=mask.device)], dim=1)
            return torch.cat([_f, _sp], dim=1), _mask
        _val_eval = ValLossEvaluator(
            dataset=_val_dataset,
            collator=OmniGen2Collator(tokenizer=text_tokenizer,
                                      max_token_len=args.data.maximum_text_tokens),
            transport=transport,
            out_path=args.val_loss.out_path,
            batch_size=args.val_loss.get("batch_size", 2),
            every=args.val_loss.get("every", 250),
            n_repeats=args.val_loss.get("n_repeats", 4),
            seed=args.val_loss.get("seed", 1234),
            encode_text=_sp_encode,
        )
        logger.info(f"[val] {len(_val_dataset)} rows every {_val_eval.every} steps, "
                    f"{_val_eval.n_repeats} draws per row")

'''

CALL = '''                if _val_eval is not None and global_step % _val_eval.every == 0:
                    try:
                        _vl = _val_eval.run(model, vae, text_encoder, freqs_cis,
                                            weight_dtype, global_step, accelerator.device)
                        logs["val_loss"] = _vl
                        logger.info("[val] step %d val_loss %.6f" % (global_step, _vl))
                    except Exception as _e:
                        logger.warning("[val] step %d failed: %r" % (global_step, _e))
'''

if "_val_eval" in src:
    print("val hook already patched")
else:
    assert src.count(A1) == 1, f"anchor A1 found {src.count(A1)} times"
    assert src.count(A2) == 1, f"anchor A2 found {src.count(A2)} times"
    open(p, "w", encoding="utf-8").write(src.replace(A1, SETUP + A1).replace(A2, A2 + CALL))
    import ast; ast.parse(open(p, encoding="utf-8").read())
    print("val hook patched")

## 9. Config
`ft_lora.yml` with `lora_ft` off, the three resolution keys at 512, and training batch size 1 with `gradient_accumulation_steps: 16` so `global_batch_size: 16` holds on one GPU while the attention mask stays an unpadded prefix. This intentionally differs from the LoRA batch-2 configuration; the resolution keys stay matched. `warmup_t` is back to the shipped 500, unlike the probe, so the schedule shape matches the LoRA arm. `checkpointing_steps` is set beyond the run length, and it is not what stops the checkpoint block here: patch S in section 6 replaces the condition line with a `False and ...` guard, so `accelerator.save_state` never runs and the prompt tensor is the only thing written. The three completed soft-prompt runs did not dead-branch it, but `checkpointing_steps` gated it out of a 3000-step run there too, so none of them wrote state either.

In [ ]:
import yaml
cfg = yaml.safe_load(open(f"{REPO}/options/ft_lora.yml"))
cfg["train"]["lora_ft"] = False
cfg["train"]["batch_size"] = 1     # batch-of-1 is unpadded, so appending the prompt after
cfg["train"]["gradient_accumulation_steps"] = 16   # text_feats keeps the mask a contiguous prefix
cfg["train"]["global_batch_size"] = 16
cfg["train"]["max_train_steps"] = STEPS
cfg["train"]["learning_rate"] = LR
cfg["train"]["gradient_checkpointing"] = True
cfg["name"] = RUN     # train.py:81 derives output_dir from conf.name; there is no train.output_dir
cfg["logger"]["log_with"] = ["tensorboard"]        # wandb would block on an interactive login
cfg["logger"]["checkpointing_steps"] = STEPS * 10  # inert: section 6 dead-branches the shipped
                                                   # save_state block; patch S writes the prompt itself
cfg["data"]["max_output_pixels"] = 512*512
cfg["data"]["max_input_pixels"] = [512*512]*4
cfg["data"]["max_side_length"] = 512
cfg["data"]["data_path"] = "data_configs/train/sp/sp.yml"
cfg["val_loss"] = {                                # 98 rows, 4 repeats, same base seed as LoRA;
                                                   # draws not matched - val batch size differs
    "data_path": "data_configs/train/sp/val_sp.yml",
    "out_path": f"{EXP}/val_loss.jsonl",
    "every": 250,
        "batch_size": 1,
    "n_repeats": 4,
    "seed": 1234,
}

OPT = f"/content/{RUN}.yml"
yaml.safe_dump(cfg, open(OPT, "w"), sort_keys=False)
print(f"lr {cfg['train']['learning_rate']} | steps {cfg['train']['max_train_steps']} | "
      f"warmup_t {cfg['train']['warmup_t']} | maximum_text_tokens {cfg['data']['maximum_text_tokens']}")
print(f"batch {cfg['train']['batch_size']} x accum {cfg['train']['gradient_accumulation_steps']} "
      f"= global {cfg['train']['global_batch_size']}")
print(f"val loss every {cfg['val_loss']['every']} steps -> {cfg['val_loss']['out_path']}")

## 10. Train
NOT resumable, by construction: patch S in section 6 dead-branches the shipped `accelerator.save_state` body, so no accelerate checkpoint is written and `resume_from_checkpoint: latest` starts clean. That is the wanted behaviour — the prompt is a bare Parameter outside `model`, so `save_state` cannot capture it, and a resume would restore weights, optimizer moments and the step counter onto a prompt reset to init. Patch S is the mitigation: a run that dies at 2000 still leaves four prompts, three of them scoring points. Run time is not yet measured at this P — re-probe before scheduling. For reference, the measured P=668 cost is 15.06 s/step for 12.6 h, and the joint sequence grows 11% to 19% at P=1,247, so the P=1,247 cost is expected to be higher, by a margin that has not been measured.

In [ ]:
_s = open(p, encoding='utf-8').read()
for _k in ('[soft_prompt]', '[sp] saved', '[sp] step', '_val_eval', '_axes_lens'):
    assert _k in _s, f'train.py is missing patch marker {_k!r} - re-run sections 5 to 8'
assert 'loss = (loss * accelerator.num_processes) / num_tokens_in_batch' in _s, \
    'train.py loss normalisation not corrected - re-run section 5a'
print('all patches present in train.py')
import time, re
t0 = time.time()
proc = subprocess.Popen(["accelerate","launch","--num_processes=1","--mixed_precision=bf16",
                         "train.py","--config",OPT],
                        cwd=REPO, stdout=subprocess.PIPE, stderr=subprocess.STDOUT,
                        text=True, bufsize=1, env={**os.environ, "WANDB_DISABLED": "true"})
LOG = f"/content/{RUN}_console.log"
with open(LOG, "w") as fh:
    for line in proc.stdout:
        fh.write(line)
        if re.search(r"\[sp\]|\[soft_prompt\]|Error|Traceback|error", line):
            print(f"[{(time.time()-t0)/3600:5.2f}h] {line.rstrip()[:150]}", flush=True)
proc.wait()
print(f"\nexit {proc.returncode} after {(time.time()-t0)/3600:.2f}h")
subprocess.run(["cp", LOG, f"{EXP}/console.log"])
if proc.returncode != 0:
    print("\n--- tail ---")
    print("".join(open(LOG, encoding="utf-8").readlines()[-40:]))
    raise RuntimeError(f"training failed with exit {proc.returncode}; see {EXP}/console.log")

## 11. Checkpoint inventory

In [ ]:
import glob
files = sorted(glob.glob(f"{EXP}/prompt-*.pt"), key=lambda f: int(re.search(r"prompt-(\d+)", f).group(1)))
print(f"{'step':>6s} {'MB':>7s} {'prompt norm':>12s} {'drift from init':>16s}")
init_n = init.norm().item()
for f in files:
    d = torch.load(f, map_location="cpu")
    n = d["soft_prompt"].float().norm().item()
    print(f"{d['step']:>6d} {os.path.getsize(f)/1e6:>7.2f} {n:>12.4f} {100*(n-init_n)/init_n:>15.1f}%")
_want = set(range(CKPT_EVERY, STEPS + 1, CKPT_EVERY))
_have = {int(re.search(r"prompt-(\d+)", f).group(1)) for f in files}
assert _have == _want, f"expected checkpoints at {sorted(_want)}, found {sorted(_have)}"
json.dump({"run": RUN, "P": P, "lr": LR, "steps": STEPS, "seed": SEED,
           "trainable": int(init.numel()), "checkpoints": [int(re.search(r"prompt-(\d+)", f).group(1)) for f in files],
           "train_rows": len(rows),
           "source_commit": COMMIT, "model_revision": MODEL_REV,
           "packages": _vers.stdout.strip()}, open(f"{EXP}/manifest.json", "w"), indent=1)
print(f"\nmanifest written to {EXP}/manifest.json")

## 12. Revert the patch

In [ ]:
subprocess.run(["git","-C",REPO,"checkout","--","train.py"], check=True)
s = open(f"{REPO}/train.py", encoding="utf-8").read()
assert "[soft_prompt]" not in s and "[sp] saved" not in s
assert "loss * accelerator.gradient_state.num_steps" in s, "loss normalisation not restored"
print("train.py reverted to HEAD")